In [245]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

train_df = pd.read_csv(
    os.path.join(os.getcwd(), "data", "raw", "train.csv")
)

test_df = pd.read_csv(
    os.path.join(os.getcwd(), "data", "raw", "test.csv")
)

all_data = pd.concat([train_df, test_df], ignore_index=True)

all_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3140 entries, 0 to 3139
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Title                3139 non-null   object
 1   Content              3139 non-null   object
 2   Target Organization  3140 non-null   object
 3   Label 1              3140 non-null   object
 4   Label 2              1547 non-null   object
 5   Label 3              759 non-null    object
 6   Label 4              241 non-null    object
 7   Label 5              57 non-null     object
dtypes: object(8)
memory usage: 196.4+ KB


In [246]:
train_df['Text'] = "Title: "+ train_df['Title'] + "Content: " +train_df['Content'] + "Target Organization: " + train_df['Target Organization']
test_df['Text'] = "Title: "+ test_df['Title'] + "Content: " +test_df['Content'] + "Target Organization: " + test_df['Target Organization']


In [247]:

test_split = 0.1
train_df, eval_df = train_test_split(
    train_df,
    test_size=test_split,
)
print(f"Number of rows in training set: {len(train_df)}")
print(f"Number of rows in test set: {len(eval_df)}")


Number of rows in training set: 2483
Number of rows in test set: 276


In [248]:
non_label_cols = ['Title','Content','Target Organization','Text']

label_columns = [col for col in train_df.columns if col not in non_label_cols]

# Create a new DataFrame containing only the selected label columns
df_labels_train = train_df[label_columns]
df_labels_test = test_df[label_columns]
df_eval = eval_df[label_columns]

# Convert the label columns to lists for each row
labels_list_train = df_labels_train.values.tolist()
labels_list_test = df_labels_test.values.tolist()
labels_list_eval = df_eval.values.tolist()

In [249]:
['1','2',label in label_columns]

['1', '2', True]

In [250]:
unique_list = []
for i, label in enumerate(label_columns):
    unique_list.append(all_data[label].unique())

unique_values = list(set(list for sublist in unique_list for list in sublist))

unique_values.remove(np.nan)

In [251]:
unique_values.remove('partnerships & alliances')
unique_values.remove('new initiatives or programs')
unique_values

['event organization',
 'other',
 'product launching & presentation',
 'patent publication',
 'investment in public company',
 'foundation',
 'expanding industry',
 'department establishment',
 'closing',
 'subsidiary establishment',
 'executive statement',
 'ipo exit',
 'service & product providing',
 'hiring',
 'expanding geography',
 'product updates',
 'executive appointment',
 'm&a',
 'new initiatives & programs',
 'company description',
 'support & philanthropy',
 'funding round',
 'clinical trial sponsorship',
 'alliance & partnership',
 'participation in an event',
 'regulatory approval',
 'article publication']

In [225]:
len(unique_values)

27

In [252]:
for name in unique_values:
    train_df[name] = 0
    test_df[name] = 0
    eval_df[name] = 0

In [253]:
unique_values

['event organization',
 'other',
 'product launching & presentation',
 'patent publication',
 'investment in public company',
 'foundation',
 'expanding industry',
 'department establishment',
 'closing',
 'subsidiary establishment',
 'executive statement',
 'ipo exit',
 'service & product providing',
 'hiring',
 'expanding geography',
 'product updates',
 'executive appointment',
 'm&a',
 'new initiatives & programs',
 'company description',
 'support & philanthropy',
 'funding round',
 'clinical trial sponsorship',
 'alliance & partnership',
 'participation in an event',
 'regulatory approval',
 'article publication']

In [227]:
def process_labels(df, labels_list):
    for x, label in enumerate(labels_list):
        for name in label:
            if name != np.nan:
                df.at[df.index[x], name] = 1

In [254]:
def process_labels(df, labels_list):
    labels = pd.Series(labels_list, index=df.index)
    dummies = (
        labels.explode()
        .dropna()
        .pipe(pd.get_dummies)
        .groupby(level=0)
        .max()
        .astype(int)
    )
    df[dummies.columns] = dummies
    return df

In [255]:
train_df = process_labels(train_df, labels_list_train)
test_df = process_labels(test_df, labels_list_test)
eval_df = process_labels(eval_df, labels_list_eval)

In [261]:
train_df

,Title,Content,Target Organization,Label 1,Label 2,Label 3,Label 4,Label 5,Text,event organization,...,company description,support & philanthropy,funding round,clinical trial sponsorship,alliance & partnership,participation in an event,regulatory approval,article publication,new initiatives or programs,partnerships & alliances
2275,MaineGeneral Health working on contingency pla...,"207-405-2502\nShare\nAUGUSTA, ME - SEPTEMBER 1...",MaineGeneral Health,executive statement,NaN,NaN,NaN,NaN,Title: MaineGeneral Health working on continge...,0,...,0,0,0,0,0,0,0,0,0,0
1686,South Carolina bowtie maker donates thousands ...,Coronavirus\nSouth Carolina bowtie maker donat...,CARE SOUTH,support & philanthropy,executive statement,NaN,NaN,NaN,Title: South Carolina bowtie maker donates tho...,0,...,0,1,0,0,0,0,0,0,0,0
507,Jet Health Lands Signal Home Health & Hospice ...,Send email\nWith its latest purchase of San An...,Jet Health,m&a,executive statement,NaN,NaN,NaN,Title: Jet Health Lands Signal Home Health & H...,0,...,0,0,0,0,0,0,0,0,0,0
1928,Guangzhou Int'l Beauty Expo 2019 CIBE,Guangzhou Intl Beauty Expo 2019 CIBE\nEXHIBITI...,ASTERASYS,product launching & presentation,NaN,NaN,NaN,NaN,Title: Guangzhou Int'l Beauty Expo 2019 CIBECo...,0,...,0,0,0,0,0,0,0,0,0,0
294,Naija Songs //,Jinmi Abduls - Abena (With Lyrics)\n3758 plays...,ABENA,other,NaN,NaN,NaN,NaN,Title: Naija Songs //Content: Jinmi Abduls - A...,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Satellite Applications Catapult and NHS organi...,Search\nSatellite Applications Catapult and NH...,Arden & GEM CSU,alliance & partnership,expanding geography,executive statement,NaN,NaN,Title: Satellite Applications Catapult and NHS...,0,...,0,0,0,0,1,0,0,0,0,0
742,"Sunny Varkey launches Tmrw, a new Learning Ope...",- Built to be accessible and affordable for sc...,Every Child,product launching & presentation,executive statement,company description,NaN,NaN,"Title: Sunny Varkey launches Tmrw, a new Learn...",0,...,1,0,0,0,0,0,0,0,0,0
618,Tataa Biocenter is partnering with Olink and I...,",\nMarch 9, 2022\n/PRNewswire/ -- Tataa Biocen...",TATAA Biocenter,alliance & partnership,product launching & presentation,executive statement,NaN,NaN,Title: Tataa Biocenter is partnering with Olin...,0,...,0,0,0,0,1,0,0,0,0,0
1143,KansasPerio welcomes new doctor,Perio and Dental Implants welcomes Dr. Mark Ha...,Kansas Perio,hiring,NaN,NaN,NaN,NaN,Title: KansasPerio welcomes new doctorContent:...,0,...,0,0,0,0,0,0,0,0,0,0


In [263]:
train_df.loc[366, 'Text']

'Title: Habitat for Humanity Buffalo dedicates \'House That Beer Built\'Content: 6:48 PM EST February 26, 2022\nUpdated:\n7:08 PM EST February 26, 2022\nBUFFALO, N.Y. After a nearly two-year delay, Habitat for Humanity Buffalo dedicated the "House That Beer Built" on Saturday.\nConstruction on the rehabbed home started in 2020. It was built with volunteers from different breweries across the Western New York region, hence the name.\nA virtual and in-person tour was held Saturday when they celebrated the home\'s completion and the King family moving in.\n"We are just so happy and excited about this process it\'s been a long time coming and now we can actually touch it," the family said.\nThe project was put on pause because of the COVID pandemic and was then faced with labor and material shortages when the project picked back up again.\nThe King family put in more than 400 hours of sweat equity to complete the home. More than 30 local bars, breweries, and distillers participated in the 

In [258]:
train_df['Text']

2275    Title: MaineGeneral Health working on continge...
1686    Title: South Carolina bowtie maker donates tho...
507     Title: Jet Health Lands Signal Home Health & H...
1928    Title: Guangzhou Int'l Beauty Expo 2019 CIBECo...
294     Title: Naija Songs //Content: Jinmi Abduls - A...
                              ...                        
495     Title: Satellite Applications Catapult and NHS...
742     Title: Sunny Varkey launches Tmrw, a new Learn...
618     Title: Tataa Biocenter is partnering with Olin...
1143    Title: KansasPerio welcomes new doctorContent:...
366     Title: Habitat for Humanity Buffalo dedicates ...
Name: Text, Length: 2483, dtype: object

In [208]:
train_df["alliance & partnership"]

2374    0
62      0
81      0
1607    1
29      0
       ..
2005    0
2702    0
2733    1
1945    0
2710    0
Name: alliance & partnership, Length: 2483, dtype: int64

In [238]:
train_df["alliance & partnership"] = (
    train_df["alliance & partnership"] +
    train_df["partnerships & alliances"]
).clip(upper=1)
test_df["alliance & partnership"] = (
    test_df["alliance & partnership"] +
    test_df["partnerships & alliances"]
).clip(upper=1)
eval_df["alliance & partnership"] = (
    eval_df["alliance & partnership"] +
    eval_df["partnerships & alliances"]
).clip(upper=1)
train_df.drop(columns=["partnerships & alliances"], inplace=True)
test_df.drop(columns=["partnerships & alliances"], inplace=True)
eval_df.drop(columns=["partnerships & alliances"], inplace=True)

KeyError: 'partnerships & alliances'

In [205]:

train_df["new initiatives & programs"] = (
    train_df["new initiatives or programs"] +
    train_df["new initiatives & programs"]
).clip(upper=1)
test_df["new initiatives & programs"] = (
    test_df["new initiatives or programs"] +
    test_df["new initiatives & programs"]
).clip(upper=1)
eval_df["new initiatives & programs"] = (
    eval_df["new initiatives or programs"] +
    eval_df["new initiatives & programs"]
).clip(upper=1)
train_df.drop(columns=["new initiatives or programs"], inplace=True)
test_df.drop(columns=["new initiatives or programs"], inplace=True)
eval_df.drop(columns=["new initiatives or programs"], inplace=True)

In [243]:
train_df['text']

KeyError: 'text'

In [244]:
train_df.iloc[0, train_df.columns.get_loc('Text')]

'Title: Rakuten plays 5G cloud-native Symphony with Robin.ioContent: Robin.io\n, the Kubernetes storage startup with a 5G edge infrastructure operations and management software stack, has been snapped up by Rakuten Symphony.\nRakuten, regarded as Japans version of Amazon, is an online retail company that owns mobile carrier Rakuten Symphony, originally Rakuten Mobile. The Symphony business unit was spun off in August 2021 and has both 4G/5G technology and services in its portfolio.\nRakuten Symphony CEO Tareq Amin said in a statement: We plan to continue to invest into Robin.ios cloud-native portfolio of products to further advance our capabilities and offer the most advanced and highly integrated cloud platform mobile operators demand.\nEdge cloud requirements are unique and critical as mobile operators transition to 5G. The next era of digital experience requires another level of performance, responsiveness and consistency that enables telecom operator and enterprise transformation t

In [218]:
train_df

,Title,Content,Target Organization,Label 1,Label 2,Label 3,Label 4,Label 5,Text
2154,Premier Foods plans to ramp up supply chain sp...,"News\nPremier Foods, which makes Mr Kipling an...",Premier Nutrition,executive statement,NaN,NaN,NaN,NaN,Title: Premier Foods plans to ramp up supply c...
534,C-CIDA identifies 3 local kits makers to ramp-...,COMMENT\nKeeping widespread testing to be the ...,DNA XPERTS,alliance & partnership,regulatory approval,NaN,NaN,NaN,Title: C-CIDA identifies 3 local kits makers t...
2331,Weak Fundamental Momentum Pushes Allot Ltd. (A...,Science\nAllot Ltd. (NASDAQ:ALLT) has a beta v...,Allot,company description,NaN,NaN,NaN,NaN,Title: Weak Fundamental Momentum Pushes Allot ...
669,Learning and development specialist (Oxford),Learning and development specialist (Oxford)\n...,AMICULUM,hiring,NaN,NaN,NaN,NaN,Title: Learning and development specialist (Ox...
2630,Singapore to test antibody treatment against C...,"June 11, 2020\n2 minutes read\nBangkok, Jun 11...",Tychan,clinical trial sponsorship,executive statement,NaN,NaN,NaN,Title: Singapore to test antibody treatment ag...
...,...,...,...,...,...,...,...,...,...
121,The Marine Corps Prepares to Test a Potential ...,Innovation\nThe Marine Corps Prepares to Test ...,CottonMouth,other,NaN,NaN,NaN,NaN,Title: The Marine Corps Prepares to Test a Pot...
674,"ViacomCBS, WarnerMedia Reportedly Mull Sale of...",French Grocer AuchanWeighs Fresh Bid for Rival...,Whole Health,m&a,company description,NaN,NaN,NaN,"Title: ViacomCBS, WarnerMedia Reportedly Mull ..."
2171,FireEye Announces Sale of FireEye Products Bus...,All cash transaction unlocks high-growth Mandi...,STG International,m&a,investment in public company,executive statement,company description,NaN,Title: FireEye Announces Sale of FireEye Produ...
436,Is The Keto Diet Healthy For Weight Loss In Ol...,Is The Keto Diet Healthy For Weight Loss In Ol...,Weight Loss Resources,article publication,NaN,NaN,NaN,NaN,Title: Is The Keto Diet Healthy For Weight Los...


In [174]:
for df in [train_df, test_df, eval_df]:
    df["labels"] = df[label_columns].values.tolist()

In [178]:
train_df['labels'] = train_df[unique_values].values.tolist()

In [179]:
train_df['labels']

317     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1235    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, ...
1402    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
2715    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
234     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                              ...                        
2272    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
361     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1202    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
1589    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
8       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: labels, Length: 2483, dtype: object

In [7]:
mapping_values = {}
for i in range(len(unique_values)):
    mapping_values.update({unique_values[i]: i})



In [8]:
mapping_values

{'event organization': 0,
 'other': 1,
 'product launching & presentation': 2,
 'patent publication': 3,
 'investment in public company': 4,
 'foundation': 5,
 'expanding industry': 6,
 'department establishment': 7,
 'closing': 8,
 'subsidiary establishment': 9,
 'executive statement': 10,
 'ipo exit': 11,
 'service & product providing': 12,
 'hiring': 13,
 'partnerships & alliances': 14,
 'expanding geography': 15,
 'product updates': 16,
 'executive appointment': 17,
 'm&a': 18,
 'new initiatives & programs': 19,
 nan: 20,
 'company description': 21,
 'support & philanthropy': 22,
 'new initiatives or programs': 23,
 'funding round': 24,
 'clinical trial sponsorship': 25,
 'alliance & partnership': 26,
 'participation in an event': 27,
 'regulatory approval': 28,
 'article publication': 29}

In [9]:
for label in label_columns:
    train_df[label] = train_df[label].map(mapping_values)
    test_df[label] = test_df[label].map(mapping_values)
    eval_df[label] = eval_df[label].map(mapping_values)

In [10]:
test_df[100:110]

,Title,Content,Target Organization,Label 1,Label 2,Label 3,Label 4,Label 5,Text
100,Prepaid AC lounge inaugurated at VZM railway s...,Vizianagaram: Divisional railway manager (DRM)...,Light Lounge,1,20,20,20,20,Title: Prepaid AC lounge inaugurated at VZM ra...
101,Nurses: Staffing levels thin at Los Robles,View Comments\nView Comments\nNurses negotiati...,Nurse Staffing,1,20,20,20,20,Title: Nurses: Staffing levels thin at Los Rob...
102,Apartment fire in Willmar sends one to the hos...,"10:29 am, Nov. 21, 2021\n\nThe Willmar Fire De...",Carris Health,1,20,20,20,20,Title: Apartment fire in Willmar sends one to ...
103,VouchForMe (IPL) Price Hits $0.0012,"Get Rating\n) by 3.6% in the 4th quarter, Hold...","SxanPro, LLC",1,20,20,20,20,Title: VouchForMe (IPL) Price Hits $0.0012Cont...
104,Why Amazon makes you click a box to redeem cou...,Attention holiday shoppers: Buy now before it ...,box-planner,1,20,20,20,20,Title: Why Amazon makes you click a box to red...
105,Super-Earths: Long-lasting radiation shields m...,The laser system at the National Ignition Faci...,Rayshield,1,20,20,20,20,Title: Super-Earths: Long-lasting radiation sh...
106,FTC Reaches Settlement With Flo Health Over Fe...,"John.McKinnon@wsj.com\nUpdated Jan. 14, 2021 7...",Flo Healthcare,1,20,20,20,20,Title: FTC Reaches Settlement With Flo Health ...
107,How to increase weapon accuracy using Octane's...,"At present, Octane is the most picked characte...",ApexHealth,1,20,20,20,20,Title: How to increase weapon accuracy using O...
108,Is increased panting cause for alarm?,"By Dr. John De Jong| Ask the Vet\nMarch 27, 20...",Coyotebio-Lab,1,20,20,20,20,Title: Is increased panting cause for alarm?Co...
109,Pivot Point Consulting Issues 2022 Healthcare ...,approximately 20% of hospitals and health clin...,IT&Care,1,20,20,20,20,Title: Pivot Point Consulting Issues 2022 Heal...


from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

# Example usage
text = "DeBERTa-small what tokenizer to use with this model"
tokens = tokenizer(text)
print(tokens)

In [11]:
from datasets import load_dataset, DatasetDict

dataset_dict = DatasetDict({
    "train": train_df,
    "test": test_df,
    "eval": eval_df})

In [17]:
dataset_dict['eval'][100:110]

,Title,Content,Target Organization,Label 1,Label 2,Label 3,Label 4,Label 5,Text
1651,Ambition is a Growth Enabler! Heres How to Lev...,11:15 AM ET\nFont Size:\nSome might think the ...,Este Medical Group,15,21,10,20,20,Title: Ambition is a Growth Enabler! Heres How...
2411,TrustedReviews Limited launches Trusted Review...,Share:\nShare\nTrustedReviews Limited has laun...,Liquiproof LABS,23,26,10,21,20,Title: TrustedReviews Limited launches Trusted...
240,Domestic Services jobs,#\nMeridian health are recruiting for experien...,StaffBank Recruitment,1,20,20,20,20,Title: Domestic Services jobsContent: #\nMerid...
362,"CareLinc Medical Equipment, CareLinc, West Mic...","Apr 21, 2021 / 01:38 PM EDT\n/\nApr 21, 2021 /...",Carelinc,1,20,20,20,20,"Title: CareLinc Medical Equipment, CareLinc, W..."
1306,Sony Honda forge strategic e-mobility alliance,Read time\n2min 20sec\nJapanese multinational ...,AllianceChicago,9,26,2,10,20,Title: Sony Honda forge strategic e-mobility a...
2195,Wasabi Technologies Becomes Official Cloud Sto...,"Wasabi Technologies\n, the hot cloud storage c...",ALung Technologies,26,10,21,20,20,Title: Wasabi Technologies Becomes Official Cl...
648,Fourth Quarter Looks Promising; Input Cost Inc...,"Jan 27, 2022, 05:31 PM\nIST (Published)\nMini\...",Symphony Corporation,10,20,20,20,20,Title: Fourth Quarter Looks Promising; Input C...
2287,Deadline looms for North East Business Awards ...,privacy notice\nThe deadline for the North Eas...,Sage Dental,10,21,20,20,20,Title: Deadline looms for North East Business ...
2484,Afghanistan army veteran turned Amazon manager...,"Published: 06:00, 24 September 2021\nMore news...",Invictus Games,10,20,20,20,20,Title: Afghanistan army veteran turned Amazon ...
1727,ChemDirect Partnership,"Posted on: October 22nd, 2019\nTedia is please...",ChemDirect,26,20,20,20,20,Title: ChemDirect PartnershipContent: Posted o...


In [18]:
!pip install Pathlib


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from transformers import DistilBertTokenizer
from Pathlib import Path
import yaml

ROOT = Path(__file__).resolve().parent.parent
config_path = ROOT / "configs/configs.yaml"

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)


tokenizer = DistilBertTokenizer.from_pretrained(cfg["model"]["name"], do_lower_case=True)
# Tokenization function
def preprocess_function(examples):
    return tokenizer(
        examples[cfg["data"]["text_column"]],
        truncation=True,
        padding="max_length",
        max_length=cfg["model"]["max_length"],
    )

# Apply tokenization to all splits
encoded_with_text = dataset_dict.map(preprocess_function, batched=True, desc="Tokenizing")

print(encoded_with_text["train"][0])

ModuleNotFoundError: No module named 'Pathlib'

In [20]:
print(train_df["Text"].isna().sum())

2


In [ ]:
train_texts = train_df['Text'].tolist()
train_labels = labels_list_train

eval_texts = test_df['Text'].tolist()
eval_labels = labels_list_test

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

train_encodings = tokenizer(train_texts, padding="max_length", truncation=True, max_length=512)
eval_encodings = tokenizer(eval_texts, padding="max_length", truncation=True, max_length=512)


/Users/conorcremin/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece installed to convert a slow tokenizer to a fast one.

In [22]:
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
config_path = ROOT / "config/config.yaml"


NameError: name '__file__' is not defined

In [31]:
import os

root = os.getcwd()
print(root)

/Users/conorcremin/Repo/Legal Document Classifier


In [33]:
from pathlib import Path
import yaml 
import os

root = os.getcwd()

#ROOT = Path(__file__).resolve().parent.parent
config_path = Path(root + "/" + str(config_path) ).resolve()
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

In [43]:
from datasets import load_from_disk
from pathlib import Path

processed_path = (Path(root) / cfg["data"]["processed_path"]).resolve()

dataset = load_from_disk(f"file://{processed_path}")

tokenized_train_dataset = dataset["train"]
print(tokenized_train_dataset)

TypeError: must be called with a dataclass type or instance

# checking dataset

In [273]:
!pip install datasets


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [277]:
from datasets import load_from_disk

dataset = load_from_disk(
    "/Users/conorcremin/Repo/Legal Document Classifier/data/processed/eu_dataset_tokenized"
)


ValueError: Protocol not known: /Users/conorcremin/Repo/Legal Document Classifier/data/processed/eu_dataset_tokenized